In [0]:
# Bug: a UDF that re-imports a repo-root module on the Spark worker.
# 'libraries' lives at the Git folder root, not next to this notebook. The UDF closure
# references gen_hello (a module-level function), so cloudpickle serializes it by
# reference and the Python worker must re-import 'libraries.gen_hello' during UDF
# deserialization. This only succeeds if the Git folder root reached the worker's
# sys.path -- otherwise fails with ModuleNotFoundError.
from libraries.gen_hello import gen_hello
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

hello_udf = udf(lambda name: gen_hello(name), StringType())

df = spark.range(1).selectExpr("'Ala' as name")
res = df.select(hello_udf("name").alias("greeting")).collect()[0]["greeting"]
print(res)
assert res == "Hello Ala", f"unexpected result from gen_hello UDF: {res}"
dbutils.notebook.exit(res)